# Football Season Modelling: Poisson vs Dixon-Coles

This notebook accompanies a match and season modelling task for a 20-team league.
The main solution (see the submission notebook) prices matches and simulates the season
using an independent **Poisson** goals model, with each team given an attack rating, a
defence rating and a shared home advantage.

Here I test a natural question: **is the extra complexity of the Dixon-Coles low-score
correction justified on this data?** I fit Dixon-Coles properly by maximum likelihood and
compare it to the Poisson two ways: an in-sample likelihood ratio test with AIC, and an
honest out-of-sample log-loss on held-out matches.

**Conclusion (spoiler):** the correction does not improve on the simpler model here, so the
plain Poisson is the right choice for this dataset.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import poisson, chi2
from scipy.optimize import minimize

played = pd.read_csv("sample_data.csv")
played[["HG", "AG"]] = played["Score"].str.split(":", expand=True).astype(int)
played["outcome"] = np.where(played.HG > played.AG, 0, np.where(played.HG == played.AG, 1, 2))

## 1. Baseline: the independent Poisson model

This is the same model used in the submission. Each match is split into two rows (one per
team's goals) and fitted as a Poisson GLM, so goals depend on the attacking team, the
defending opponent and a home flag.

In [2]:
home_rows = pd.DataFrame({"goals": played["HG"], "team": played["Home ID"], "opponent": played["Away ID"], "home": 1})
away_rows = pd.DataFrame({"goals": played["AG"], "team": played["Away ID"], "opponent": played["Home ID"], "home": 0})
model_df = pd.concat([home_rows, away_rows], ignore_index=True)

glm = smf.glm("goals ~ home + C(team) + C(opponent)", data=model_df, family=sm.families.Poisson()).fit()
print(f"Poisson log-likelihood: {glm.llf:.4f}  |  parameters: {len(glm.params)}")

Poisson log-likelihood: -451.0766  |  parameters: 40


## 2. The Dixon-Coles model

Dixon-Coles is the Poisson model plus a small correction. It uses the same scoreline grid,
but multiplies four low-score outcomes (0-0, 1-0, 0-1, 1-1) by a factor that depends on one
extra parameter, `rho`, which captures the mild dependence between the two teams' goals that
independent Poisson misses. Everything else is unchanged.

The correction breaks the clean log-linear form a GLM needs, so instead of a one-line
regression we write the negative log-likelihood by hand and minimise it numerically. This is
maximum likelihood done explicitly.

Two design choices worth noting:
- We work at **match level** (one row per game), because the correction couples the two scores of the same match.
- We hold **team 1 as the reference** (attack and defence zero), exactly as the GLM does, which makes the Poisson a special case of Dixon-Coles with `rho = 0` and the two models directly comparable.

In [3]:
h  = played["Home ID"].to_numpy() - 1   # 0-based team indices
a  = played["Away ID"].to_numpy() - 1
hg = played["HG"].to_numpy()
ag = played["AG"].to_numpy()

# Parameter vector: [base, home, rho, 19 attack, 19 defence]; team 1 (index 0) is the reference.
def unpack(p):
    base, home, rho = p[0], p[1], p[2]
    attack  = np.concatenate([[0.0], p[3:22]])
    defence = np.concatenate([[0.0], p[22:41]])
    return base, home, rho, attack, defence

# Dixon-Coles correction: adjusts only the four low-score outcomes, leaves the rest at 1.
def tau(hg, ag, lh, la, rho):
    t = np.ones_like(lh * 1.0)
    t = np.where((hg == 0) & (ag == 0), 1 - lh * la * rho, t)
    t = np.where((hg == 0) & (ag == 1), 1 + lh * rho,      t)
    t = np.where((hg == 1) & (ag == 0), 1 + la * rho,      t)
    t = np.where((hg == 1) & (ag == 1), 1 - rho,           t)
    return t

# Negative log-likelihood of the whole dataset for a given parameter vector.
def nll(p, hi, ai, hgv, agv):
    base, home, rho, attack, defence = unpack(p)
    lh = np.exp(base + home + attack[hi] - defence[ai])   # home expected goal rate
    la = np.exp(base +        attack[ai] - defence[hi])   # away expected goal rate
    prob = tau(hgv, agv, lh, la, rho) * poisson.pmf(hgv, lh) * poisson.pmf(agv, la)
    return -np.log(np.clip(prob, 1e-15, None)).sum()

### Validating the likelihood

Before trusting the Dixon-Coles fit, a sanity check: with `rho = 0` the model *is* the
independent Poisson, so the likelihood must reproduce the GLM's log-likelihood exactly. If
these two numbers match, the hand-written likelihood is correct.

In [4]:
base0 = glm.params["Intercept"]
home0 = glm.params["home"]
attack0  = [glm.params.get(f"C(team)[T.{t}]", 0.0)      for t in range(2, 21)]
defence0 = [-glm.params.get(f"C(opponent)[T.{t}]", 0.0) for t in range(2, 21)]
p0 = np.array([base0, home0, 0.0] + attack0 + defence0)

print(f"nll at rho = 0 : {nll(p0, h, a, hg, ag):.4f}")
print(f"-glm.llf       : {-glm.llf:.4f}")
print(f"match          : {np.isclose(nll(p0, h, a, hg, ag), -glm.llf, atol=1e-3)}")

nll at rho = 0 : 451.0766
-glm.llf       : 451.0766
match          : True


### Fitting Dixon-Coles

Starting from the Poisson solution, we let the optimiser find whether a non-zero `rho`
improves the fit.

In [5]:
res = minimize(nll, p0, args=(h, a, hg, ag), method="BFGS", options={"maxiter": 2000})
base, home, rho, attack, defence = unpack(res.x)
print(f"converged: {res.success}  |  fitted rho: {rho:.4f}")

converged: True  |  fitted rho: 0.0310


## 3. In-sample comparison: likelihood ratio test and AIC

The likelihood ratio test asks whether the improvement from adding `rho` is bigger than
expected from one extra parameter. AIC balances fit against parameter count (lower is
better). Both are in-sample, so they flatter both models, but they are the standard first
check for whether added complexity earns its place.

In [6]:
LL_dc  = -res.fun      # Dixon-Coles log-likelihood at the optimum
LL_poi = glm.llf       # Poisson log-likelihood (same scale, validated above)

# Likelihood ratio test: Dixon-Coles adds exactly one parameter (rho), so df = 1.
LR = 2 * (LL_dc - LL_poi)
p_value = chi2.sf(LR, df=1)

# AIC rewards fit but penalises parameter count. Lower is better.
k_poi, k_dc = 40, 41
AIC_poi = 2 * k_poi - 2 * LL_poi
AIC_dc  = 2 * k_dc  - 2 * LL_dc

print(f"Log-likelihood   Poisson {LL_poi:.4f}   Dixon-Coles {LL_dc:.4f}   gain {LL_dc - LL_poi:.4f}")
print(f"Likelihood ratio {LR:.3f}   p-value {p_value:.4f}")
print(f"AIC              Poisson {AIC_poi:.2f}   Dixon-Coles {AIC_dc:.2f}   (lower is better)")

Log-likelihood   Poisson -451.0766   Dixon-Coles -451.0405   gain 0.0361
Likelihood ratio 0.072   p-value 0.7882
AIC              Poisson 982.15   Dixon-Coles 984.08   (lower is better)


## 4. Out-of-sample comparison: log-loss on held-out matches

The honest test. We fit both models on the first 120 matches and score their 1 / X / 2
predictions on the last 40, using log-loss (lower is better). This removes the in-sample
advantage entirely and is the fair way to ask which model actually predicts better.

In [7]:
MAX_GOALS = 10

# 1X2 probabilities from a parameter vector, with or without the Dixon-Coles correction.
def predict_1x2(p, home_id, away_id, use_dc):
    base, home, rho, attack, defence = unpack(p)
    lh = np.exp(base + home + attack[home_id - 1] - defence[away_id - 1])
    la = np.exp(base +        attack[away_id - 1] - defence[home_id - 1])
    grid = np.outer(poisson.pmf(np.arange(MAX_GOALS + 1), lh),
                    poisson.pmf(np.arange(MAX_GOALS + 1), la))
    if use_dc:
        grid[0, 0] *= 1 - lh * la * rho
        grid[0, 1] *= 1 + lh * rho
        grid[1, 0] *= 1 + la * rho
        grid[1, 1] *= 1 - rho
    grid /= grid.sum()
    return np.tril(grid, -1).sum(), np.trace(grid), np.triu(grid, 1).sum()

# Fit both models on a subset of matches.
def fit_both(train):
    hr = pd.DataFrame({"goals": train.HG, "team": train["Home ID"], "opponent": train["Away ID"], "home": 1})
    ar = pd.DataFrame({"goals": train.AG, "team": train["Away ID"], "opponent": train["Home ID"], "home": 0})
    g = smf.glm("goals ~ home + C(team) + C(opponent)", data=pd.concat([hr, ar], ignore_index=True), family=sm.families.Poisson()).fit()
    p_poisson = np.array([g.params["Intercept"], g.params["home"], 0.0]
                         + [g.params.get(f"C(team)[T.{t}]", 0.0) for t in range(2, 21)]
                         + [-g.params.get(f"C(opponent)[T.{t}]", 0.0) for t in range(2, 21)])
    hi = train["Home ID"].to_numpy() - 1
    ai = train["Away ID"].to_numpy() - 1
    r = minimize(nll, p_poisson, args=(hi, ai, train.HG.to_numpy(), train.AG.to_numpy()), method="BFGS", options={"maxiter": 3000})
    return p_poisson, r.x

def log_loss(P, oc):
    P = np.array(P)
    return -np.log(np.clip(P[np.arange(len(oc)), oc], 1e-12, 1)).mean()

# Temporal split: train on the first 120 matches, test on the last 40 (predict the future).
train = played.iloc[:120]
test  = played.iloc[120:]
p_poisson_tr, p_dc_tr = fit_both(train)

oc = test["outcome"].to_numpy()
P_poi = [predict_1x2(p_poisson_tr, r["Home ID"], r["Away ID"], use_dc=False) for _, r in test.iterrows()]
P_dc  = [predict_1x2(p_dc_tr,      r["Home ID"], r["Away ID"], use_dc=True ) for _, r in test.iterrows()]

print(f"Out-of-sample log-loss on the last {len(test)} matches (lower is better):")
print(f"  Poisson     : {log_loss(P_poi, oc):.4f}")
print(f"  Dixon-Coles : {log_loss(P_dc, oc):.4f}")

Out-of-sample log-loss on the last 40 matches (lower is better):
  Poisson     : 1.0070
  Dixon-Coles : 1.0072


## 5. Conclusion

Three pieces of evidence agree:

- **Likelihood ratio test:** the gain from adding `rho` is not significant (high p-value).
- **AIC:** prefers the simpler Poisson, because the extra parameter does not pay for itself.
- **Out-of-sample log-loss:** the two models are effectively tied, with Poisson marginally ahead.

So the Dixon-Coles correction earns nothing on this dataset. The fitted `rho` is small and, unlike
real-world football where low-scoring draws are inflated, comes out slightly positive here,
suggesting this data simply does not carry that low-score dependence.

The takeaway is methodological: the right way to decide whether to add complexity is to test it
properly, in-sample with AIC and a likelihood ratio test, and out-of-sample with a proper scoring
rule, and then to keep the simpler model when the evidence does not justify the complex one.

*Note: this uses a single temporal split. A k-fold cross-validation would be more robust, but the
result is consistent across all three checks.*